# Data Loader Fuzzy Tests

This section performs lightweight randomized checks of all functions in `data_loader.py`.
The tests use short date windows and handle network-related failures gracefully.

In [1]:
from __future__ import annotations

import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from data_loader import (
    _normalize_tickers,
    align_returns_and_daily_rate,
    annual_to_daily_rate,
    compute_daily_simple_returns,
    download_adj_close_prices,
    fetch_fred_annual_rate,
    load_market_data,
)

random.seed(7)
np.random.seed(7)

TICKER_POOL = ["SPY", "QQQ", "IWM", "TLT", "GLD", "EFA", "VTI"]
FRED_POOL = ["SOFR", "DFF", "DTB3"]

end = datetime.utcnow().date() - timedelta(days=2)
start = end - timedelta(days=220)

picked_tickers = random.sample(TICKER_POOL, k=3)
picked_fred = random.choice(FRED_POOL)

print("Selected tickers:", picked_tickers)
print("Selected FRED series:", picked_fred)
print("Date window:", start, "->", end)


def check(name: str, condition: bool, details: str = ""):
    if condition:
        print(f"[PASS] {name}")
    else:
        raise AssertionError(f"[FAIL] {name}. {details}")

Selected tickers: ['IWM', 'QQQ', 'TLT']
Selected FRED series: DTB3
Date window: 2025-08-17 -> 2026-03-25


/Users/giacomomaggiore/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# 1) _normalize_tickers
norm = _normalize_tickers([" spy ", "qqq", "", "tlt"])
check("normalize_tickers returns cleaned uppercase symbols", norm == ["SPY", "QQQ", "TLT"], str(norm))

# 2) download_adj_close_prices + 3) compute_daily_simple_returns
prices = download_adj_close_prices(picked_tickers, start=start, end=end)
check("prices dataframe is non-empty", not prices.empty)
check("prices columns match requested tickers", set(prices.columns) == set(picked_tickers), str(prices.columns.tolist()))
check("prices index is datetime", isinstance(prices.index, pd.DatetimeIndex))

returns = compute_daily_simple_returns(prices)
check("returns dataframe is non-empty", not returns.empty)
check("returns shape follows prices", returns.shape[1] == prices.shape[1])
check("returns has finite values", np.isfinite(returns.to_numpy(dtype=float)).all())

# 4) fetch_fred_annual_rate + 5) annual_to_daily_rate
annual_rate = fetch_fred_annual_rate(picked_fred, start=start, end=end)
check("annual rate is non-empty", not annual_rate.empty)
check("annual rate has datetime index", isinstance(annual_rate.index, pd.DatetimeIndex))

daily_rate = annual_to_daily_rate(annual_rate, trading_days=252, is_percent=True)
check("daily rate is non-empty", not daily_rate.empty)
check("daily rate is decimal-scale (reasonable magnitude)", daily_rate.dropna().abs().quantile(0.95) < 0.01)

# 6) align_returns_and_daily_rate
aligned_returns, aligned_rate = align_returns_and_daily_rate(returns, daily_rate)
check("aligned returns/rate share identical index", aligned_returns.index.equals(aligned_rate.index))
check("aligned index equals trading-day returns index", aligned_returns.index.equals(returns.index))
check("aligned rate has some valid observations", aligned_rate.notna().any())

# 7) load_market_data end-to-end
pipeline_returns, pipeline_rate = load_market_data(
    tickers=picked_tickers,
    fred_series=picked_fred,
    start=start,
    end=end,
    fred_is_percent=True,
)
check("pipeline returns non-empty", not pipeline_returns.empty)
check("pipeline rate non-empty", not pipeline_rate.empty)
check("pipeline outputs share same index", pipeline_returns.index.equals(pipeline_rate.index))
check("pipeline index is datetime", isinstance(pipeline_returns.index, pd.DatetimeIndex))

print("\nAll fuzzy data_loader checks passed.")

[PASS] normalize_tickers returns cleaned uppercase symbols
[PASS] prices dataframe is non-empty
[PASS] prices columns match requested tickers
[PASS] prices index is datetime
[PASS] returns dataframe is non-empty
[PASS] returns shape follows prices
[PASS] returns has finite values
[PASS] annual rate is non-empty
[PASS] annual rate has datetime index
[PASS] daily rate is non-empty
[PASS] daily rate is decimal-scale (reasonable magnitude)
[PASS] aligned returns/rate share identical index
[PASS] aligned index equals trading-day returns index
[PASS] aligned rate has some valid observations
[PASS] pipeline returns non-empty
[PASS] pipeline rate non-empty
[PASS] pipeline outputs share same index
[PASS] pipeline index is datetime

All fuzzy data_loader checks passed.
